# 02 · Suy luận (inference)

Chạy pipeline `datamate` trên một split và ghi `submission.jsonl`: nạp shop → view và lớp công nợ → 8 mẫu SQL thường + 4 mẫu thinking cho câu
khó → chạy, sửa lỗi, bỏ phiếu. Cờ lấy từ `scripts/flags.sh` (bản đã nộp, public 0.683) và tự chỉnh theo GPU.

**Phần cứng:** một GPU bf16 (Ampere trở lên) từ 24 GB cho mỗi tiến trình; nhiều GPU thì mỗi GPU một phần shop. Ada / Hopper / Blackwell chạy fp8
như bản đã nộp, Ampere (A10, A100…) chạy bf16 với context ngắn hơn. T4 / P100 không chạy được model này.

**Dữ liệu:** split của BTC trong `data/` (`data/train`, `data/public_test`, cạnh bộ SFT `data/data`) hoặc `DATAMATE_DATA`.

**Thời gian:** model nạp vài phút; `LIMIT = 10` để thử; cả 1000 câu public khoảng 3,5 giờ trên một GPU 24 GB.

| Mục đích | `SPLIT` | `IDS` | `REUSE_LAYERS` | `LORA` | `SAVE_PROMPTS` |
|---|---|---|---|---|---|
| Bản nộp public | `"public_test"` | `None` | `True` | adapter đã chọn | tùy |
| Lượt có nhãn để dựng dữ liệu SFT (`04`) | `"train"` | `"splits/rft_train24.txt"` | `False` | `"sql_rft_v3"` | `True` |
| Đo adapter mới trên holdout (`05` → `03`) | `"train"` | `"splits/holdout600.txt"` | `False` | `"runs/lora/qwen35_sft_a6000/adapter"` | tùy |
| Mốc so sánh trên holdout | `"train"` | `"splits/holdout600.txt"` | `False` | `"sql_rft_v3"` | tùy |

In [ ]:
# Setup: tìm checkout của repo này, nạp datamate.notebook, phát hiện dữ liệu, GPU, thư mục model và runs.
#   local   mở notebook trong notebooks/ của checkout (hoặc đặt biến môi trường DATAMATE_REPO), không cần cài thêm.
#   Kaggle  đính kèm bản sao checkout làm dataset (Add data) cùng dữ liệu BTC (dataset riêng tư), bật Internet để cài gói.
#   Colab   chép checkout vào MyDrive/datamate/code, dữ liệu BTC vào MyDrive/datamate/data.
# Kaggle / Colab: code được chép sang ổ ghi được (input của Kaggle chỉ đọc, Drive chậm); kết quả ghi vào env.runs, được giữ sau phiên.
import os, shutil, sys
from pathlib import Path

MARKER = "datamate/notebook.py"


def find_checkout() -> Path:
    if os.environ.get("DATAMATE_REPO"):
        return Path(os.environ["DATAMATE_REPO"]).resolve()
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / MARKER).exists():
            return c.resolve()
    if Path("/kaggle/input").exists():
        found, work = sorted(p.parents[1] for p in Path("/kaggle/input").glob(f"**/{MARKER}")), Path("/tmp/datamate")
    elif "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive")
        found, work = [p for p in [Path("/content/drive/MyDrive/datamate/code")] if (p / MARKER).exists()], Path("/content/datamate")
    else:
        found, work = [], None
    if not found:
        raise FileNotFoundError("không tìm thấy checkout datamate: mở notebook trong notebooks/ của repo, hoặc đặt DATAMATE_REPO")
    if not (work / MARKER).exists():
        shutil.copytree(found[0], work, ignore=shutil.ignore_patterns(".git", "runs", "models", ".venv*", "__pycache__"))
    return work


root = find_checkout()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("llm")
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

In [ ]:
SPLIT = "public_test"        # public_test | train | private_test (thư mục trong data/)
LIMIT = 10                   # N câu đầu (None = tất cả)
IDS = None                   # tệp id câu hỏi, ví dụ "splits/holdout600.txt" hoặc "splits/rft_train24.txt"
REUSE_LAYERS = True          # chỉ public: dùng lại view và lớp công nợ của lượt 0.683 (configs/public_v9). False = soạn mới (như private)
LORA = "sql_rft_v3"          # adapter trong adapters/, hoặc adapter từ 05_train_lora, ví dụ "runs/lora/qwen35_sft_a6000/adapter"
SAVE_PROMPTS = True          # lưu prompt trong trace (cần để dựng dữ liệu SFT từ lượt chạy train)
OUT_NAME = None              # None = nb_<split>_<tên adapter>

In [ ]:
env.split(SPLIT)                                                        # dữ liệu BTC của split này
lora_arg = env.path(LORA) if env.path(LORA).exists() else LORA            # đường dẫn (runs/..., tuyệt đối) hoặc tên trong adapters/
lora_name = Path(LORA).parent.name if Path(LORA).name == "adapter" else Path(LORA).name
out = env.runs / (OUT_NAME or f"nb_{SPLIT}_{lora_name}")
if env.can_run_llm:
    model, lora = env.base_model(), env.adapter(lora_arg)
    flags, engine = env.inference_flags()
    n = len(env.gpus)
    layers = ["--views-from", "configs/public_v9"] if REUSE_LAYERS and SPLIT == "public_test" else []
    pick = (["--limit", str(LIMIT)] if LIMIT else []) + (["--ids", env.path(IDS)] if IDS else []) + (["--save-prompts"] if SAVE_PROMPTS else [])
    make_cmd = lambda i: [env.python, "-m", "datamate.predict", "--data", env.data / SPLIT, "--output", out / f"shard{i}", "--shard", f"{i}/{n}",
                          "--model", model, "--lora", lora, "--fewshot", "assets/fewshot_bank.jsonl", *flags, "--engine", engine, *layers, *pick]
    print(f"{n} GPU process(es) -> {out}\nadapter: {lora}\nflags: {' '.join(flags)}")
else:   # T4 / P100 hoặc CPU: dùng lượt chạy y hệt đã chạy sẵn trên GPU 24 GB (public_test, LIMIT = 10), nếu checkout có
    try:
        env.use_saved("nb_" + SPLIT, out)
    except RuntimeError as e:
        print("không có GPU bf16 >= 24 GB và không có kết quả chạy sẵn:", e)

In [ ]:
if env.can_run_llm:
    env.run_shards(make_cmd, n, out)

In [ ]:
from datamate.submission import check, merge_shards

if list(out.glob("shard*/trace.jsonl")):
    merge_shards(out, env.data / SPLIT / "questions.jsonl")
    errors = check(out / "submission.jsonl", env.data / SPLIT / "questions.jsonl")
    print("format OK" if not errors else errors[:5])
else:
    print(f"chưa có lượt chạy trong {out}")

## Kết quả

Số câu đã trả lời theo shop, vài câu mẫu cùng đáp án và tỷ lệ đồng thuận (`conf`). Với `SPLIT = "train"` có thêm điểm theo đáp án train.

In [ ]:
trace = [r for p in sorted(out.glob("shard*/trace.jsonl")) for r in read(p)]
df = pd.DataFrame([{"shop": r["business_id"], "answered": r["final"]["type"] != "abstain", "conf": r["conf"]} for r in trace])
if len(df):
    display(df.groupby("shop").agg(questions=("answered", "size"), answered=("answered", "sum"), mean_conf=("conf", "mean")).round(2))
pd.DataFrame([{"question": r["question"][:90], "final": json.dumps(r["final"], ensure_ascii=False)[:80], "conf": round(r["conf"], 2)} for r in trace[:5]])

In [ ]:
if trace and (env.data / SPLIT / "answers.jsonl").exists():
    from datamate import scoring
    gold = {a["id"]: a for a in read(env.data / SPLIT / "answers.jsonl")}
    report, _ = scoring.evaluate({r["id"]: gold[r["id"]] for r in trace}, {r["id"]: r["final"] for r in trace},
                                 {q["id"]: q for q in read(env.data / SPLIT / "questions.jsonl")})
    print({k: report[k] for k in ("N", "correct", "wrong", "abstain", "PA", "precision")})